# Comparing Git Merge Strategies Across Branch Topologies

A reference guide to merge commit, rebase, squash, and cherry-pick — when each strategy fits, how they reshape history, and what to watch for in CI and release workflows.

## Purpose

Compare the four primary Git merge strategies — merge commit, rebase, squash, and cherry-pick — across common branch topologies (feature branches, release branches, and main). Understand the trade-offs in history readability, traceability, and CI compatibility so that teams can choose the right strategy for each workflow.

## When to use

| Strategy | Best suited for | Avoid when |
|---|---|---|
| Merge commit | Preserving full branch history; feature branches that need traceability to their origin | Linear history is a strict requirement; the extra merge commit adds noise |
| Rebase | Keeping a linear history on long-running branches; preparing commits for clean review | The branch has already been pushed and shared with other collaborators |
| Squash | Consolidating a messy feature branch into a single atomic commit on main | Individual commit granularity matters for bisect or rollback |
| Cherry-pick | Applying a specific fix from one branch to another without merging the full branch | The fix depends on a chain of preceding commits that are not also cherry-picked

## Prerequisites

- Git installed and configured (user.name and user.email)
- A local repository with at least two branches (e.g., `main` and a feature branch)
- Familiarity with basic Git commands: `git branch`, `git checkout`, `git log --oneline --graph`

## Steps

### 1. Merge commit

Creates a new merge commit that ties together the histories of two branches. The full branch history is preserved, and the merge commit serves as a clear integration point.

In [ ]:
# Create a feature branch, make commits, then merge with --no-ff
git checkout -b feature/add-cart main
git commit --allow-empty -m "feat: add shopping cart component"
git commit --allow-empty -m "feat: add cart validation logic"
git checkout main
git merge --no-ff feature/add-cart -m "merge: integrate feature/add-cart"

### 2. Rebase

Replays commits from the current branch onto the tip of another branch, producing a linear history. Use `--onto` to control the target and base precisely.

In [ ]:
# Rebase feature branch onto updated main before merging
git checkout feature/add-cart
git rebase main
# If conflicts arise, resolve them then continue
git add <resolved-files>
git rebase --continue
# Fast-forward merge after rebase keeps history linear
git checkout main
git merge feature/add-cart

### 3. Squash

Combines all commits from a branch into a single commit on the target branch. The individual commit history is lost, but the working tree changes are preserved.

In [ ]:
# Squash-merge a feature branch into main
git checkout main
git merge --squash feature/add-cart
git commit -m "feat: add shopping cart with validation"

### 4. Cherry-pick

Applies the changes introduced by a specific commit to the current branch without merging the full branch history. Useful for backporting fixes.

In [ ]:
# Cherry-pick a specific fix commit from hotfix onto main
git checkout main
git cherry-pick <commit-hash>
# Cherry-pick a range of commits
git cherry-pick <start-hash>^..<end-hash>

### Branch topology comparison

| Topology | Merge commit | Rebase | Squash | Cherry-pick |
|---|---|---|---|---|
| Feature → main | Preserves branch context; merge commit marks integration | Linear history; loses branch context unless merge commit is kept | Single commit on main; no branch traceability | Apply individual fixes without full branch merge |
| Release → main | Preserves release boundary; useful for audit trails | Linearizes release commits; can obscure release boundaries | Consolidates release into one commit; loses individual release commits | Backport specific fixes from release to main |
| Hotfix → main | Preserves hotfix context with a merge commit | Linearizes hotfix; clean but loses branch separation | Single hotfix commit; simple and clean | Apply hotfix fix to multiple branches simultaneously

## Verify

After any merge strategy, confirm the result with these commands:

```bash
# View the commit graph to check history shape
git log --oneline --graph --all

# Verify the diff between branches is empty (all changes integrated)
git diff main..feature/add-cart

# Check that CI passes on the merged result
git push origin main
# Confirm CI pipeline status in your CI tool

## Common errors

| Error | Cause | Fix |
|---|---|---|
| `CONFLICT (content)` during rebase | Two branches modified the same lines | Resolve conflicts in each file, then `git add` and `git rebase --continue` |
| `fatal: refusing to merge unrelated histories` | Merging two repos with no common ancestor | Add `--allow-unrelated-histories` to the merge command |
| `error: failed to push some refs` after rebase | Remote history diverged from rebased local history | Force-push with `git push --force-with-lease` (only on personal branches, never on shared branches) |
| Cherry-pick introduces duplicate changes | The commit was already applied | Use `git cherry-pick --no-commit` and inspect changes before committing, or check `git log` for the commit first |
| Squash merge loses commit messages | `--squash` stages changes but does not auto-commit | Write a descriptive commit message that summarizes the entire feature